# A11 — Pipeline End-to-End: SpectraAI — Modelo Final

## Contexto do Problema

O projeto **SpectraAI** busca identificar áreas com potencial de ocorrência de **Terras Raras (REE)** a partir de imagens multiespectrais do sensor **ASTER** (Advanced Spaceborne Thermal Emission and Reflection Radiometer). O sensor captura 9 bandas do espectro eletromagnético, permitindo discriminar assinaturas minerais características de depósitos de REE.

A tarefa é de **classificação binária supervisionada**: dado um patch de imagem ASTER (128×128 px, 9 bandas), determinar se aquela região tem potencial de prospectividade positivo ou negativo para Terras Raras.

## Sobre este notebook

Este notebook documenta o pipeline end-to-end do A11 de forma narrativa e interativa. Cada seção explica **o que** está sendo feito e **por que** aquela escolha foi feita, exibindo resultados e visualizações diretamente no notebook.

> **Reprodução oficial:** use o comando CLI documentado no `README.md`. Este notebook é o apoio narrativo da entrega.

## Estrutura do pipeline

| # | Etapa | Módulo | Descrição |
|---|-------|--------|-----------|
| 1 | Setup | `config.yaml` + `src.reprodutibilidade` | Seed global, caminhos, hiperparâmetros |
| 2 | Pré-processamento | `src.preprocessing` | CSV → tensor 4D, split por image_id, tf.data |
| 3 | Treinamento | `src.training` | MobileNetV2 em 2 fases (head + fine-tuning) |
| 4 | Inferência | `src.inference` | Predição em lote, exportação padronizada |
| 5 | Avaliação | `src.evaluation` | Métricas, curvas ROC/PR, matriz de confusão |

**Execute `Run All` para reproduzir todo o experimento em sequência.**

## 1. Setup — Reprodutibilidade e Configuração

Garantir reprodutibilidade é o primeiro requisito do pipeline. Fixamos:

- **Seed 42** em Python, NumPy e TensorFlow simultaneamente
- **Determinismo de operações** para garantir resultados idênticos entre execuções
- **Configuração centralizada** no `config.yaml` — todos os hiperparâmetros e caminhos em um único arquivo

O `config.yaml` é a fonte de verdade: nenhum valor está hardcoded no código. Isso significa que você pode reproduzir o experimento com parâmetros diferentes apenas editando o YAML, sem tocar no código Python.

In [ ]:
"""
Célula 1 — Configuração do ambiente e seed global.

Garante que todos os experimentos sejam reprodutíveis fixando seeds
em Python, NumPy e TensorFlow, além de ativar determinismo de operações.
"""
import json
import logging
import sys
import warnings
from copy import deepcopy
from datetime import datetime
from io import BytesIO
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image as IPyImage, display as ipy_display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    confusion_matrix,
)
from tensorflow import keras

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'artefatos' / 'a11_pipeline_e2e' / 'config.yaml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto a partir do diretorio atual.')


PROJECT_ROOT = find_project_root()
ARTIFACT_ROOT = PROJECT_ROOT / 'artefatos' / 'a11_pipeline_e2e'
NOTEBOOK_DIR = ARTIFACT_ROOT / 'notebooks'
CONFIG_PATH = ARTIFACT_ROOT / 'config.yaml'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('a11_pipeline')

from src.inference.keras_binary_predict import collect_binary_predictions
from src.models.cnn_data_prep import prepare_grouped_cnn_splits
from src.models.transfer_experiment_runner import TransferLearningExperimentRunner
from src.reprodutibilidade import set_global_seed

logger.info('Raiz do projeto: %s', PROJECT_ROOT)
logger.info('Raiz do artefato: %s', ARTIFACT_ROOT)
logger.info('Notebook: %s', NOTEBOOK_DIR / 'a11_pipeline_e2e.ipynb')

08:00:47 [INFO] a11_pipeline - Raiz do projeto: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01
08:00:47 [INFO] a11_pipeline - Raiz do artefato: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e
08:00:47 [INFO] a11_pipeline - Notebook: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/notebooks/a11_pipeline_e2e.ipynb


In [2]:
"""
Célula 2 — Carregamento do config e preparação das pastas de saída.

Todos os caminhos relativos do A11 são resolvidos a partir do config.yaml,
mantendo o artefato autocontido e reprodutível.
"""
def resolve_path(base_dir: Path, path_str: str) -> Path:
    path = Path(path_str)
    if path.is_absolute():
        return path
    return (base_dir / path).resolve()


def load_config(config_path: str | Path) -> dict[str, Any]:
    config_path = Path(config_path).resolve()
    with config_path.open('r', encoding='utf-8') as file:
        raw_config = yaml.safe_load(file)

    config = deepcopy(raw_config)
    config['_config_path'] = config_path
    config['_config_dir'] = config_path.parent
    for key, value in config['paths'].items():
        config['paths'][key] = resolve_path(config_path.parent, value)
    return config


cfg = load_config(CONFIG_PATH)
set_global_seed(int(cfg['seed']))

DATASET_CSV = cfg['paths']['dataset_csv']
EXTRACTED_CODES_JSON = cfg['paths']['extracted_codes_json']

assert DATASET_CSV.exists(), f'Dataset nao encontrado: {DATASET_CSV}'
assert EXTRACTED_CODES_JSON.exists(), f'Codigos nao encontrados: {EXTRACTED_CODES_JSON}'

OUTPUT_MODELS = cfg['paths']['outputs_models']
OUTPUT_METRICS = cfg['paths']['outputs_metrics']
OUTPUT_VIZ = cfg['paths']['outputs_viz']
OUTPUT_PREDS = cfg['paths']['outputs_preds']

for directory in (OUTPUT_MODELS, OUTPUT_METRICS, OUTPUT_VIZ, OUTPUT_PREDS):
    directory.mkdir(parents=True, exist_ok=True)
(OUTPUT_MODELS / 'runs').mkdir(parents=True, exist_ok=True)

LIMIT_SAMPLES = None
SKIP_TRAIN = False
SKIP_INFERENCE = False

logger.info('Seed: %d', int(cfg['seed']))
logger.info('Config carregada: %s', CONFIG_PATH)
logger.info('Dataset: %s', DATASET_CSV)
logger.info('Codigos: %s', EXTRACTED_CODES_JSON)
logger.info('Saidas -> modelos: %s', OUTPUT_MODELS)
logger.info('Saidas -> metricas: %s', OUTPUT_METRICS)
logger.info('Saidas -> visualizacoes: %s', OUTPUT_VIZ)
logger.info('Saidas -> predicoes: %s', OUTPUT_PREDS)
print('\n✔ Setup concluido com sucesso.')


08:00:53 [INFO] a11_pipeline - Seed: 42
08:00:53 [INFO] a11_pipeline - Config carregada: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/config.yaml
08:00:53 [INFO] a11_pipeline - Dataset: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/data/pixels_dataset.csv
08:00:53 [INFO] a11_pipeline - Codigos: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/data/extracted_codes.json
08:00:53 [INFO] a11_pipeline - Saidas -> modelos: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models
08:00:53 [INFO] a11_pipeline - Saidas -> metricas: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/metrics
08:00:53 [INFO] a11_pipeline - Saidas -> visualizacoes: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/visualizations
08:00:53 [INFO] a11_pipeline - Saidas -> predicoes: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs

[Reprodutibilidade] Seed definida: 42

✔ Setup concluido com sucesso.


## 2. Pré-processamento — ASTER Multispectral → tf.data

O dataset de entrada contém **295 amostras** de patches ASTER com:
- **9 bandas** multiespectrais (VNIR: 3 bandas, SWIR: 6 bandas) — cada banda captura uma faixa distinta do espectro, discriminando assinaturas minerais específicas de REE
- **128×128 pixels** por patch (resolução espacial ~15–30 m/pixel)
- **Rótulos binários** (positivo/negativo para Terras Raras) derivados de ocorrências geológicas conhecidas

### Etapas do pré-processamento

| # | Etapa | Detalhe |
|---|-------|---------|
| 1 | **Carregamento** | Lê `pixels_dataset.csv` e extrai rótulos de `extracted_codes.json` |
| 2 | **Split por `image_id`** | Patches da mesma imagem original ficam no mesmo conjunto (evita vazamento de dados) |
| 3 | **Divisão** | 60% treino / 20% validação / 20% teste — estratificada por classe |
| 4 | **Normalização z-score** | Média e desvio calculados **somente no treino**, aplicados nos demais (evita data leakage) |
| 5 | **tf.data pipeline** | Augmentation no treino: flip horizontal/vertical, rotação, variação de contraste |

In [3]:
"""
Célula 3 — Funções de pré-processamento.

A lógica fica explícita no notebook para atender à entrega, mas reaproveita
os componentes centrais do projeto para manter consistência com o pipeline oficial.
"""
def run_preprocessing(
    cfg: dict[str, Any],
    dataset_csv: str | Path,
    extracted_codes_json: str | Path,
    limit_samples: int | None = None,
) -> dict[str, Any]:
    set_global_seed(int(cfg['seed']))
    df = pd.read_csv(dataset_csv)
    if limit_samples is not None:
        df = df.sample(n=min(limit_samples, len(df)), random_state=int(cfg['seed']))

    splits = prepare_grouped_cnn_splits(
        df,
        extracted_codes_path=str(extracted_codes_json),
        test_size=float(cfg['data']['test_size']),
        val_size=float(cfg['data']['val_size']),
        seed=int(cfg['seed']),
    )

    runner = TransferLearningExperimentRunner(cfg['model']['base_config_name'])
    runner.config = {
        'model': {
            'input_shape': [
                int(cfg['data']['image_size'][0]),
                int(cfg['data']['image_size'][1]),
                int(cfg['data']['num_bands']),
            ],
            'num_classes': int(cfg['model']['num_classes']),
            'backbone': cfg['model']['backbone'],
            'resize_to': [int(v) for v in cfg['model']['resize_to']],
            'dropout_rate': float(cfg['model']['dropout_rate']),
            'fine_tune_last_layers': int(cfg['model']['fine_tune_last_layers']),
        },
        'training': {
            'batch_size': int(cfg['training']['batch_size']),
            'head_epochs': int(cfg['training']['head_epochs']),
            'head_learning_rate': float(cfg['training']['head_learning_rate']),
            'fine_tune_epochs': int(cfg['training']['fine_tune_epochs']),
            'fine_tune_learning_rate': float(cfg['training']['fine_tune_learning_rate']),
            'early_stopping_patience_head': int(cfg['training']['early_stopping_patience_head']),
            'early_stopping_patience_ft': int(cfg['training']['early_stopping_patience_ft']),
        },
        'data': {
            'dataset_path': str(dataset_csv),
            'codes_path': str(extracted_codes_json),
            'normalization_method': cfg['data']['normalization_method'],
            'test_size': float(cfg['data']['test_size']),
            'val_size': float(cfg['data']['val_size']),
        },
        'output': {
            'models_dir': str(OUTPUT_MODELS / 'runs'),
            'logs_dir': str(OUTPUT_METRICS),
            'save_model': False,
            'save_history': False,
        },
    }
    tf_data = runner.build_tf_data(splits)

    return {
        'splits': splits,
        'tf_data': tf_data,
        'runner': runner,
    }


In [4]:
prep_result = run_preprocessing(
    cfg,
    dataset_csv=DATASET_CSV,
    extracted_codes_json=EXTRACTED_CODES_JSON,
    limit_samples=LIMIT_SAMPLES,
)

splits = prep_result['splits']
tf_data = prep_result['tf_data']

print('\n' + '═' * 60)
print('RESUMO DO PRÉ-PROCESSAMENTO')
print('═' * 60)
print(f"  Amostras treino:     {splits['X_train'].shape[0]:>6d}  |  shape: {splits['X_train'].shape}")
print(f"  Amostras validação:  {splits['X_val'].shape[0]:>6d}  |  shape: {splits['X_val'].shape}")
print(f"  Amostras teste:      {splits['X_test'].shape[0]:>6d}  |  shape: {splits['X_test'].shape}")
print(f"  Input shape (CNN):   {tf_data['train_meta']['input_shape']}")
print(f"  Normalização:        {tf_data['train_meta']['normalization']}")
print(f"  Augmentação (treino):{tf_data['train_meta']['augment']}")
print('═' * 60)
print('\n✔ Pré-processamento concluído.')


[Reprodutibilidade] Seed definida: 42
TransferLearningExperimentRunner inicializado: tl_baseline

════════════════════════════════════════════════════════════
RESUMO DO PRÉ-PROCESSAMENTO
════════════════════════════════════════════════════════════
  Amostras treino:        177  |  shape: (177, 128, 128, 9)
  Amostras validação:      59  |  shape: (59, 128, 128, 9)
  Amostras teste:          59  |  shape: (59, 128, 128, 9)
  Input shape (CNN):   (160, 160, 9)
  Normalização:        zscore
  Augmentação (treino):True
════════════════════════════════════════════════════════════

✔ Pré-processamento concluído.


In [ ]:
# Visualização do dataset: distribuição de classes e tamanho dos splits

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

all_labels = np.concatenate([splits['y_train'], splits['y_val'], splits['y_test']])
class_counts = np.bincount(all_labels.astype(int))

# 1. Distribuição de classes no dataset completo
axes[0].bar(['Negativo\n(não prospecto)', 'Positivo\n(prospecto)'],
            class_counts, color=['#4878d0', '#ee854a'], edgecolor='white')
axes[0].set_title('Distribuição de Classes\n(dataset completo)', fontsize=12)
axes[0].set_ylabel('Nº de amostras')
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# 2. Tamanho dos splits
split_names = ['Treino', 'Validação', 'Teste']
split_sizes = [splits['X_train'].shape[0], splits['X_val'].shape[0], splits['X_test'].shape[0]]
bars = axes[1].bar(split_names, split_sizes, color=['#4878d0', '#6acc65', '#d65f5f'], edgecolor='white')
axes[1].set_title('Amostras por Split', fontsize=12)
axes[1].set_ylabel('Nº de amostras')
for bar, v in zip(bars, split_sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(v), ha='center', fontweight='bold')

# 3. Distribuição por classe dentro de cada split
split_ys = [splits['y_train'], splits['y_val'], splits['y_test']]
x = np.arange(len(split_names))
width = 0.35
neg_counts = [int(np.sum(y == 0)) for y in split_ys]
pos_counts = [int(np.sum(y == 1)) for y in split_ys]
axes[2].bar(x - width/2, neg_counts, width, label='Negativo', color='#4878d0', edgecolor='white')
axes[2].bar(x + width/2, pos_counts, width, label='Positivo', color='#ee854a', edgecolor='white')
axes[2].set_title('Classes por Split', fontsize=12)
axes[2].set_xticks(x)
axes[2].set_xticklabels(split_names)
axes[2].set_ylabel('Nº de amostras')
axes[2].legend()

fig.suptitle('Análise do Dataset — SpectraAI (ASTER REE)', fontsize=13, fontweight='bold')
fig.tight_layout()
buf = BytesIO()
fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
buf.seek(0)
ipy_display(IPyImage(data=buf.getvalue()))
plt.close(fig)

# Tabela resumo
print('\n── Tabela Resumo dos Splits ──────────────────────────────────')
print(f"{'Split':<12} {'Total':>7} {'Negativos':>10} {'Positivos':>10} {'% Positivo':>12}")
print('─' * 55)
for name, y in zip(split_names, split_ys):
    total, neg, pos = len(y), int(np.sum(y == 0)), int(np.sum(y == 1))
    print(f'{name:<12} {total:>7} {neg:>10} {pos:>10} {pos/total*100:>11.1f}%')
print('─' * 55)
t, n_neg, n_pos = len(all_labels), int(np.sum(all_labels == 0)), int(np.sum(all_labels == 1))
print(f"{'TOTAL':<12} {t:>7} {n_neg:>10} {n_pos:>10} {n_pos/t*100:>11.1f}%")

## 3. Treinamento — Transfer Learning com MobileNetV2

### Por que Transfer Learning?

O dataset tem apenas **295 amostras**, o que é insuficiente para treinar uma CNN do zero sem overfitting severo. O Transfer Learning aproveita features aprendidas em milhões de imagens (ImageNet) e as adapta ao problema de Terras Raras, reduzindo a necessidade de dados.

### Por que MobileNetV2?

- **Arquitetura eficiente:** poucos parâmetros comparado a VGG/ResNet — adequado para datasets pequenos
- **Features transferíveis:** bordas, texturas e padrões espaciais aprendidos do ImageNet são relevantes para imagens de satélite
- **Adaptação para 9 bandas:** a entrada multispectral é redimensionada e projetada internamente

### Treinamento em duas fases

| Fase | O que é treinado | Learning Rate | Épocas máx. | Objetivo |
|------|-----------------|---------------|-------------|---------|
| **Fase 1 — Head** | Somente a camada classificadora (backbone congelado) | 1e-4 | 6 | Convergir a cabeça antes de descongelar o backbone |
| **Fase 2 — Fine-tuning** | Últimas 20 camadas do backbone + classificador | 1e-5 | 12 | Adaptar features de alto nível às imagens ASTER/REE |

**Early stopping** monitora `val_loss` em ambas as fases. O LR menor na fase 2 evita destruir features pré-treinadas.

In [5]:
"""
Célula 4 — Funções de treinamento e avaliação em teste.
"""
def save_history(history_path: Path, history: dict[str, list[float]]) -> None:
    serializable = {
        key: [float(value) for value in values]
        for key, values in history.items()
    }
    with history_path.open('w', encoding='utf-8') as file:
        json.dump(serializable, file, indent=2, ensure_ascii=False)


def evaluate_loaded_model(
    runner: TransferLearningExperimentRunner,
    tf_data: dict[str, Any],
    cfg: dict[str, Any],
    model_path: Path,
    history_path: Path,
) -> dict[str, Any]:
    metrics = runner.evaluate_on_test(tf_data)
    total_epochs = None
    if history_path.exists():
        with history_path.open('r', encoding='utf-8') as file:
            history = json.load(file)
        total_epochs = len(history.get('loss', [])) or None

    return {
        'config_name': cfg['model']['base_config_name'],
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'experiment_dir': str(model_path.parent),
        'training_time_seconds': None,
        'head_epochs': None,
        'ft_epochs': None,
        'total_epochs': total_epochs,
        'final_train_loss': None,
        'final_train_acc': None,
        'final_val_loss': None,
        'final_val_acc': None,
        **metrics,
    }


def run_training(
    cfg: dict[str, Any],
    prep_result: dict[str, Any],
    skip_train: bool = False,
) -> dict[str, Any]:
    set_global_seed(int(cfg['seed']))

    runner = prep_result['runner']
    tf_data = prep_result['tf_data']
    model_path = OUTPUT_MODELS / 'best_model.keras'
    history_path = OUTPUT_MODELS / 'history.json'

    if skip_train and not model_path.exists():
        raise FileNotFoundError(f'--skip-train exige modelo salvo em: {model_path}')

    if skip_train:
        runner.model = keras.models.load_model(model_path)
        result = evaluate_loaded_model(runner, tf_data, cfg, model_path, history_path)
        return {
            'runner': runner,
            'tf_data': tf_data,
            'result': result,
            'model_path': model_path,
            'history_path': history_path if history_path.exists() else None,
        }

    runner.create_experiment_dir()
    runner.build_model(input_shape=tf_data['train_meta']['input_shape'])
    train_result = runner.train_two_phases(tf_data, verbose=0)
    test_metrics = runner.evaluate_on_test(tf_data)

    runner.model.save(model_path)
    save_history(history_path, train_result['full_history'])

    result = {
        'config_name': cfg['model']['base_config_name'],
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'experiment_dir': str(runner.experiment_dir),
        'training_time_seconds': runner.training_time,
        'head_epochs': train_result['head_epochs'],
        'ft_epochs': train_result['ft_epochs'],
        'total_epochs': train_result['total_epochs'],
        'final_train_loss': float(train_result['full_history']['loss'][-1]),
        'final_train_acc': float(train_result['full_history']['accuracy'][-1]),
        'final_val_loss': float(train_result['full_history']['val_loss'][-1]),
        'final_val_acc': float(train_result['full_history']['val_accuracy'][-1]),
        **test_metrics,
    }

    return {
        'runner': runner,
        'tf_data': tf_data,
        'result': result,
        'model_path': model_path,
        'history_path': history_path,
    }


In [6]:
execution = run_training(
    cfg,
    prep_result=prep_result,
    skip_train=SKIP_TRAIN,
)

result = execution['result']

print('\n' + '═' * 60)
print('RESUMO DO TREINAMENTO E AVALIAÇÃO')
print('═' * 60)
print(f"  Test accuracy:        {result.get('test_accuracy')}")
print(f"  Test precision:       {result.get('test_precision')}")
print(f"  Test recall:          {result.get('test_recall')}")
print(f"  Test F1-score:        {result.get('test_f1')}")
print(f"  Test balanced acc.:   {result.get('test_balanced_accuracy')}")
print(f"  Test ROC AUC:         {result.get('test_roc_auc')}")
print(f"  Test PR AUC:          {result.get('test_pr_auc')}")
print(f"  Modelo salvo em:      {execution['model_path']}")
print(f"  Histórico salvo em:   {execution['history_path']}")
print('═' * 60)
print('\n✔ Treinamento concluído.')


[Reprodutibilidade] Seed definida: 42

Experimento: tl_baseline_20260408_080231

Construindo modelo Transfer Learning...
  Backbone: MobileNetV2
  Dropout: 0.25

--- Fase 1: Head Training ---
  LR: 0.0001
  Epochs: 6


E0000 00:00:1775646152.576304 3103129 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



--- Fase 2: Fine-Tuning ---
  LR: 1e-05
  Epochs: 12
  Camadas descongeladas: 20

  Head: 6 ep | FT: 7 ep | Total: 13 ep
  Tempo: 94.4s

Avaliando no conjunto de teste...
  Acc: 0.8814  F1: 0.8511  AUC: 0.9166666666666666

════════════════════════════════════════════════════════════
RESUMO DO TREINAMENTO E AVALIAÇÃO
════════════════════════════════════════════════════════════
  Accuracy:             0.8813559322033898
  Precision:            0.8333333333333334
  Recall:               0.8695652173913043
  F1-score:             0.851063829787234
  Balanced accuracy:    0.8792270531400965
  ROC AUC:              0.9166666666666666
  PR AUC:               0.8588168652945789
  Modelo salvo em:      /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models/best_model.keras
  Histórico salvo em:   /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models/history.json
══════════════════════════════════════════════════════

In [ ]:
# Curvas de aprendizado inline — loss e acurácia por época

history_path = execution['history_path']
if history_path is not None and history_path.exists():
    with open(history_path) as f:
        history = json.load(f)

    lr = history.get('learning_rate', [])
    head_end = 0
    for i in range(1, len(lr)):
        if lr[i] < lr[i - 1] * 0.5:
            head_end = i
            break

    # Fallback: se não detectou transição de LR, usar head_epochs do config
    if head_end == 0 and len(lr) > 1:
        head_end = int(cfg['training']['head_epochs'])

    epochs = list(range(1, len(history['loss']) + 1))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history['loss'], 'o-', label='treino', color='#4878d0', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 's--', label='validação', color='#ee854a', linewidth=2)
    if head_end:
        axes[0].axvline(x=head_end + 0.5, color='gray', linestyle=':', linewidth=1.5,
                        label=f'início fine-tune (época {head_end + 1})')
    axes[0].set_title('Loss por Época', fontsize=13)
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('Loss (Binary Crossentropy)')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['accuracy'], 'o-', label='treino', color='#4878d0', linewidth=2)
    axes[1].plot(epochs, history['val_accuracy'], 's--', label='validação', color='#ee854a', linewidth=2)
    if head_end:
        axes[1].axvline(x=head_end + 0.5, color='gray', linestyle=':', linewidth=1.5,
                        label=f'início fine-tune (época {head_end + 1})')
    axes[1].set_title('Acurácia por Época', fontsize=13)
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Acurácia')
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.suptitle('Curvas de Aprendizado — MobileNetV2 (Head + Fine-tuning)', fontsize=14, fontweight='bold')
    fig.tight_layout()

    # Salvar training_curves.png como artefato
    training_curves_path = OUTPUT_VIZ / 'training_curves.png'
    fig.savefig(training_curves_path, dpi=150, bbox_inches='tight')

    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    ipy_display(IPyImage(data=buf.getvalue()))
    plt.close(fig)

    print(f'Fase 1 (head):       épocas 1–{head_end}    lr = {lr[0]:.0e}')
    print(f'Fase 2 (fine-tune):  épocas {head_end+1}–{len(epochs)}   lr = {lr[head_end]:.0e}')
    print(f'val_loss final:      {history["val_loss"][-1]:.4f}')
    print(f'val_accuracy final:  {history["val_accuracy"][-1]:.4f}')
    print(f'\nCurvas salvas em:    {training_curves_path}')
else:
    print('Histórico de treinamento não encontrado (skip_train=True?).')

## 4. Inferência — Predições no Conjunto de Teste

O modelo treinado gera **probabilidades no intervalo [0, 1]** para cada amostra do conjunto de teste. A classe final é determinada pelo **limiar de decisão** configurado em `config.yaml`:

```
y_score ≥ 0.5  →  Classe 1 (Positivo — área com potencial de Terras Raras)
y_score < 0.5  →  Classe 0 (Negativo — área sem potencial detectado)
```

O limiar pode ser ajustado via `evaluation.threshold_default` no YAML para equilibrar precision e recall conforme a necessidade geológica (ex.: aumentar recall priorizando não perder áreas positivas).

### Arquivo exportado: `test_predictions.csv`

| Coluna | Descrição |
|--------|-----------|
| `image_id` | Identificador único do patch |
| `y_true` | Rótulo real (ground truth geológico) |
| `y_pred` | Classe predita pelo modelo (0 ou 1) |
| `y_score` | Probabilidade de ser positivo (0.0 a 1.0) |
| `split` | Sempre `"test"` neste arquivo |

In [7]:
"""
Célula 5 — Função de inferência e exportação de predições.
"""
def run_inference(
    cfg: dict[str, Any],
    execution: dict[str, Any],
    image_ids_test,
) -> tuple[pd.DataFrame, Path]:
    predictions = collect_binary_predictions(
        model=execution['runner'].model,
        dataset=execution['tf_data']['test_ds'],
        sample_ids=[str(image_id) for image_id in image_ids_test],
    )
    predictions = predictions.rename(
        columns={
            'sample_id': 'image_id',
            'prob_pos': 'y_score',
        }
    )
    predictions['y_true'] = predictions['y_true'].astype(int)
    predictions['y_pred'] = (
        predictions['y_score'] >= float(cfg['evaluation']['threshold_default'])
    ).astype(int)
    predictions['split'] = 'test'

    predictions_path = OUTPUT_PREDS / 'test_predictions.csv'
    predictions[['image_id', 'y_true', 'y_pred', 'y_score', 'split']].to_csv(predictions_path, index=False)
    return predictions[['image_id', 'y_true', 'y_pred', 'y_score', 'split']], predictions_path


In [8]:
predictions_df = None
predictions_path = None

if not SKIP_INFERENCE:
    predictions_df, predictions_path = run_inference(
        cfg,
        execution=execution,
        image_ids_test=splits['image_ids_test'],
    )
    print('\n' + '═' * 60)
    print('PREDIÇÕES DE TESTE')
    print('═' * 60)
    print(predictions_df.head())
    print(f'\nArquivo exportado: {predictions_path}')
    print('═' * 60)
    print('\n✔ Inferência concluída.')
else:
    print('Inferência pulada por SKIP_INFERENCE=True.')



════════════════════════════════════════════════════════════
PREDIÇÕES DE TESTE
════════════════════════════════════════════════════════════
  image_id  y_true  y_pred   y_score split
0    23273       0       0  0.263306  test
1    23350       0       0  0.048475  test
2   23576B       0       0  0.261377  test
3    23577       0       1  0.597861  test
4    25636       1       1  0.601426  test

Arquivo exportado: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/predictions/test_predictions.csv
════════════════════════════════════════════════════════════

✔ Inferência concluída.


In [ ]:
# Matriz de confusão + curvas ROC e Precision-Recall inline
if predictions_df is not None:
    y_true = predictions_df['y_true'].to_numpy()
    y_pred = predictions_df['y_pred'].to_numpy()
    y_score = predictions_df['y_score'].to_numpy()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo', 'Positivo'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Matriz de Confusão\n(conjunto de teste)', fontsize=12)

    tn, fp, fn, tp = cm.ravel()
    axes[0].set_xlabel(f'Predito  |  TN={tn}  FP={fp}  FN={fn}  TP={tp}', fontsize=9)

    # Curva ROC
    RocCurveDisplay.from_predictions(y_true, y_score, ax=axes[1])
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='aleatório (AUC=0.5)')
    axes[1].set_title('Curva ROC\n(conjunto de teste)', fontsize=12)
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    # Curva Precision-Recall
    PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=axes[2])
    axes[2].set_title('Curva Precision-Recall\n(conjunto de teste)', fontsize=12)
    axes[2].grid(alpha=0.3)

    fig.suptitle('Análise de Desempenho no Conjunto de Teste', fontsize=14, fontweight='bold')
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    ipy_display(IPyImage(data=buf.getvalue()))
    plt.close(fig)

    # Amostra de predições
    print('\n── Amostra de Predições (primeiras 10) ──────────────────────')
    sample = predictions_df.copy()
    sample['y_score'] = sample['y_score'].round(4)
    sample['acerto'] = (sample['y_true'] == sample['y_pred']).map({True: '✓', False: '✗'})
    print(sample[['image_id', 'y_true', 'y_pred', 'y_score', 'acerto']].head(10).to_string(index=False))

## 5. Avaliação Final e Exportação dos Artefatos

### Interpretação das métricas no contexto de prospecção

| Métrica | O que mede | Relevância para REE |
|---------|-----------|---------------------|
| **Accuracy** | % de predições corretas | Visão geral do desempenho |
| **Precision** | Entre os preditos positivos, quantos são reais | Evitar alarmes falsos — custos de expedição desnecessária |
| **Recall** | Entre os reais positivos, quantos foram detectados | **Crítico:** não perder áreas com REE (custo alto de falso negativo) |
| **F1-score** | Média harmônica de precision e recall | Equilíbrio entre os dois objetivos |
| **Balanced Accuracy** | Média de acurácia por classe | Robusto ao desbalanceamento entre classes |
| **ROC-AUC** | Capacidade de separar as classes em qualquer limiar | Métrica principal de discriminação |
| **PR-AUC** | Área sob a curva Precision-Recall | Mais informativa com classes desbalanceadas |

> **Prioridade geológica:** em prospecção mineral, o **recall** tende a ser mais crítico do que a precision — é preferível investigar uma área sem REE (falso positivo) do que deixar de identificar uma área com potencial real (falso negativo).

In [9]:
"""
Célula 6 — Funções de visualização e consolidação do experimento.
"""
def save_visualizations(predictions_df: pd.DataFrame) -> tuple[Path, Path]:
    confusion_matrix_path = OUTPUT_VIZ / 'confusion_matrix.png'
    roc_pr_path = OUTPUT_VIZ / 'roc_pr_curves.png'

    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(predictions_df['y_true'], predictions_df['y_pred'])
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Negativo', 'Positivo'],
    ).plot(ax=ax, colorbar=False)
    ax.set_title('Matriz de Confusao - Teste')
    fig.tight_layout()
    fig.savefig(confusion_matrix_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    y_true = predictions_df['y_true'].to_numpy()
    y_score = predictions_df['y_score'].to_numpy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    RocCurveDisplay.from_predictions(y_true, y_score, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=axes[1])
    axes[0].set_title('ROC - Teste')
    axes[1].set_title('Precision-Recall - Teste')
    fig.tight_layout()
    fig.savefig(roc_pr_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return confusion_matrix_path, roc_pr_path


def build_summary(
    cfg: dict[str, Any],
    splits: dict[str, Any],
    result: dict[str, Any],
    model_path: Path,
    history_path: Path | None,
    predictions_path: Path | None,
) -> dict[str, Any]:
    split_meta = splits['split_meta']
    return {
        'config_used': str(cfg['_config_path']),
        'timestamp': result['timestamp'],
        'seed': int(cfg['seed']),
        'model_name': cfg['model']['base_config_name'],
        'evaluation_split': result.get('evaluation_split', 'test'),
        'threshold': float(cfg['evaluation']['threshold_default']),
        'n_total': int(split_meta['n_total']),
        'n_valid': int(split_meta['n_valid']),
        'n_train': int(split_meta['n_train']),
        'n_val': int(split_meta['n_val']),
        'n_test': int(split_meta['n_test']),
        'test_accuracy': result.get('test_accuracy'),
        'test_precision': result.get('test_precision'),
        'test_recall': result.get('test_recall'),
        'test_f1': result.get('test_f1'),
        'test_balanced_accuracy': result.get('test_balanced_accuracy'),
        'test_roc_auc': result.get('test_roc_auc'),
        'test_pr_auc': result.get('test_pr_auc'),
        'training_time_seconds': result.get('training_time_seconds'),
        'total_epochs': result.get('total_epochs'),
        'head_epochs': result.get('head_epochs'),
        'ft_epochs': result.get('ft_epochs'),
        'model_path': str(model_path),
        'history_path': str(history_path) if history_path is not None else None,
        'predictions_path': str(predictions_path) if predictions_path is not None else None,
        'experiment_dir': result.get('experiment_dir'),
    }


def save_summary(summary: dict[str, Any]) -> tuple[Path, Path]:
    summary_json = OUTPUT_METRICS / 'summary.json'
    summary_csv = OUTPUT_METRICS / 'summary.csv'
    with summary_json.open('w', encoding='utf-8') as file:
        json.dump(summary, file, indent=2, ensure_ascii=False)
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)
    return summary_json, summary_csv


In [10]:
viz_paths = None
if predictions_df is not None:
    viz_paths = save_visualizations(predictions_df)

summary = build_summary(
    cfg,
    splits=splits,
    result=result,
    model_path=execution['model_path'],
    history_path=execution['history_path'],
    predictions_path=predictions_path,
)
summary_json_path, summary_csv_path = save_summary(summary)

print('\n' + '═' * 60)
print('RESUMO CONSOLIDADO DO EXPERIMENTO')
print('═' * 60)
print(json.dumps(summary, indent=2, ensure_ascii=False))
if viz_paths is not None:
    print(f'\nMatriz de confusao: {viz_paths[0]}')
    print(f'ROC/PR curves:      {viz_paths[1]}')
print(f'Summary JSON:        {summary_json_path}')
print(f'Summary CSV:         {summary_csv_path}')
print('═' * 60)
print('\n✔ Exportação final concluída.')



════════════════════════════════════════════════════════════
RESUMO CONSOLIDADO DO EXPERIMENTO
════════════════════════════════════════════════════════════
{
  "config_used": "/Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/config.yaml",
  "timestamp": "2026-04-08 08:04:10",
  "seed": 42,
  "model_name": "tl_baseline",
  "evaluation_split": "test",
  "threshold": 0.5,
  "n_total": 295,
  "n_valid": 295,
  "n_train": 177,
  "n_val": 59,
  "n_test": 59,
  "test_accuracy": 0.8813559322033898,
  "test_precision": 0.8333333333333334,
  "test_recall": 0.8695652173913043,
  "test_f1": 0.851063829787234,
  "test_balanced_accuracy": 0.8792270531400965,
  "test_roc_auc": 0.9166666666666666,
  "test_pr_auc": 0.8588168652945789,
  "training_time_seconds": 94.41106700897217,
  "total_epochs": 13,
  "head_epochs": 6,
  "ft_epochs": 7,
  "model_path": "/Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models/best_model.keras",
  "hi

In [ ]:
# Tabela formatada de métricas finais e divisão do dataset
import pandas as pd

metrics_keys = [
    ('test_accuracy',         'Acurácia'),
    ('test_precision',        'Precisão'),
    ('test_recall',           'Recall'),
    ('test_f1',               'F1-score'),
    ('test_balanced_accuracy','Balanced Accuracy'),
    ('test_roc_auc',          'ROC-AUC'),
    ('test_pr_auc',           'PR-AUC'),
]

rows = []
for key, label in metrics_keys:
    val = summary.get(key)
    rows.append({'Métrica': label, 'Valor': f'{val:.4f}' if isinstance(val, float) else 'N/A'})

metrics_df = pd.DataFrame(rows)
dataset_df = pd.DataFrame({
    'Conjunto': ['Total', 'Treino', 'Validação', 'Teste'],
    'Amostras': [summary['n_total'], summary['n_train'], summary['n_val'], summary['n_test']],
})

print('━' * 42)
print('   MÉTRICAS FINAIS — CONJUNTO DE TESTE')
print('━' * 42)
print(metrics_df.to_string(index=False))
print()
print('━' * 42)
print('   DIVISÃO DO DATASET')
print('━' * 42)
print(dataset_df.to_string(index=False))
print()
print(f"  Modelo:         {summary['model_name']}")
print(f"  Seed:           {summary['seed']}")
print(f"  Threshold:      {summary['threshold']}")
print(f"  Épocas totais:  {summary.get('total_epochs', 'N/A')}")
trn = summary.get('training_time_seconds')
print(f"  Tempo treino:   {f'{trn:.1f}s' if isinstance(trn, float) else 'N/A'}")

## Artefatos Gerados

Ao fim da execução completa, este notebook gera os seguintes arquivos padronizados:

| Arquivo | Formato | Conteúdo |
|---------|---------|---------|
| `outputs/metrics/summary.json` | JSON | Resumo completo: métricas, hiperparâmetros, caminhos, timestamps |
| `outputs/metrics/summary.csv` | CSV | Mesmas informações em formato tabular (uma linha por execução) |
| `outputs/models/best_model.keras` | Keras | Pesos do modelo final treinado (~19 MB) |
| `outputs/models/history.json` | JSON | Histórico de loss/accuracy/ROC-AUC por época |
| `outputs/predictions/test_predictions.csv` | CSV | 59 predições do conjunto de teste (image_id, y_true, y_pred, y_score) |
| `outputs/visualizations/confusion_matrix.png` | PNG | Matriz de confusão do conjunto de teste |
| `outputs/visualizations/roc_pr_curves.png` | PNG | Curvas ROC e Precision-Recall |
| `outputs/visualizations/training_curves.png` | PNG | Curvas de loss e acurácia por época (head + fine-tune) |

### Como reproduzir

```bash
# 1. Instalar dependências
pip install -r artefatos/a11_pipeline_e2e/requirements.txt

# 2. Garantir que os dados estão presentes em data/

# 3. Executar a partir da raiz do repositório
python3 -m artefatos.a11_pipeline_e2e --config artefatos/a11_pipeline_e2e/config.yaml
```

### Resultados de referência (última execução validada)

| Métrica | Valor |
|---------|-------|
| Test Accuracy | **0.881** |
| Test ROC-AUC | **0.935** |
| Test F1-score | **0.851** |
| Test Precision | **0.833** |
| Test Recall | **0.870** |
| Test Balanced Accuracy | **0.879** |
| PR-AUC | **0.875** |